AOI properties to compare:

- [ ] Land cover breakdown (to see potential for confusion between mangroves & dense vegetation)
- [ ] Slope class distribution (to see potential for confusion from topographic shadow effects)
- [ ] Mangrove patch fragmentation (to see effect of patch sizes on accuracy)
- [ ] Climate type (Taylor chart showing Type 1, 2, 3? determines which months to best get imagery)

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from odc.stac import stac_load, configure_s3_access
import pystac_client
from dask.distributed import Client

import json
from pathlib import Path

In [15]:
def extract_aoi_geom(fp):
    with open(fp, 'r') as f:
        geojson_dict = json.load(f)

    geom_dict = geojson_dict['features'][0]['geometry']

    return geom_dict

In [23]:
def extract_lulc_aoi(geom_dict, catalog, collection_id, year=2021):
    search = catalog.search(
        collections=[collection_id],
        intersects=geom_dict,
        datetime=f'{year}-01-01'
    )

    items = search.item_collection()

    if len(items) == 0:
        raise ValueError('Search in catalog returned no results')

    ds = stac_load(
        items,
        resolution=10,
        crs='utm',
        chunks={'x': 1_024, 'y': 1_024}
    )

    return ds

In [2]:
DIR_IMAGES = Path("C:\\Users\\remot\\Documents\\Jerome\\03_Independent_Research\\20260630 Mangrove and Seagrass Monitoring\\03_Data_management\\01_Raw_data\\Rasters")
fps_images = list(DIR_IMAGES.glob("*_PlanetScope8band_*"))

# Only use a specific date for Hinatuan
fps_images = [fp for fp in fps_images if "Hinatuan" not in fp.stem or "20210729" in fp.stem]

for fp in fps_images:
    print(fp)

C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\01_Raw_data\Rasters\Busuanga-Palawan_PlanetScope8band_20240421.zip
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\01_Raw_data\Rasters\Calauag-Quezon_PlanetScope8band_20250804
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\01_Raw_data\Rasters\DumaranIsland-Palawan_PlanetScope8band_20220524
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\01_Raw_data\Rasters\Hinatuan-SurigaoDelSur_PlanetScope8band_20210729
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\01_Raw_data\Rasters\Infanta-Quezon_PlanetScope8band_20210903
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Mo

In [3]:
DIR_AOIs = Path().resolve().parent / 'output'
fps_aoi = list(DIR_AOIs.glob("*_AOI_MangroveSeagrassMapping*"))

aoi_names = [fp.stem.split("-")[0] for fp in fps_images]
fps_aoi = [fp for fp in fps_aoi if fp.stem.split("_")[0] in aoi_names]

for fp in fps_aoi:
    print(fp)

C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\03_Scripts\output\Busuanga_AOI_MangroveSeagrassMapping.geojson
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\03_Scripts\output\Calauag_AOI_MangroveSeagrassMapping.geojson
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\03_Scripts\output\DumaranIsland_AOI_MangroveSeagrassMapping.geojson
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\03_Scripts\output\Hinatuan_AOI_MangroveSeagrassMapping.geojson
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\03_Scripts\output\Infanta_AOI_MangroveSeagrassMapping.geojson
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring

In [4]:
client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 5
Total threads: 20,Total memory: 15.63 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:61929,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:61961,Total threads: 4
Dashboard: http://127.0.0.1:61962/status,Memory: 3.13 GiB
Nanny: tcp://127.0.0.1:61932,


1) Land cover breakdown (using ESRI 10-meter land cover)

In [5]:
api_url = 'https://planetarycomputer.microsoft.com/api/stac/v1/'
collection_id = 'io-lulc'

catalog = pystac_client.Client.open(api_url)

In [6]:
configure_s3_access(aws_unsigned=True)

In [17]:
search = catalog.search(
    collections=[collection_id],
    intersects=extract_aoi_geom(fps_aoi[0]),
    datetime='2021-01-01'
)

items = search.item_collection()
items

In [18]:
ds = stac_load(
    items,
    resolution=10,
    crs='utm',
    chunks={'x': 1024, 'y': 1024},
)

ds

<xarray.Dataset> Size: 59GB
Dimensions:      (y: 103126, x: 143808, time: 1)
Coordinates:
  * y            (y) float64 825kB 1.82e+06 1.82e+06 ... 7.884e+05 7.884e+05
  * x            (x) float64 1MB -5.297e+05 -5.297e+05 ... 9.083e+05 9.083e+05
  * time         (time) datetime64[us] 8B 2020-06-01
    spatial_ref  int32 4B 32651
Data variables:
    data         (time, y, x) float32 59GB dask.array<chunksize=(1, 1024, 1024), meta=np.ndarray>

In [26]:
ds_per_aoi = {}

for fp in fps_aoi:
    aoi_name = fp.stem.split("_")[0]
    aoi_geom = extract_aoi_geom(fp)
    
    ds = extract_lulc_aoi(aoi_geom, catalog, collection_id)
    ds_per_aoi[aoi_name] = ds

ds_per_aoi['RoxasCity']

<xarray.Dataset> Size: 31GB
Dimensions:      (y: 100942, x: 77037, time: 1)
Coordinates:
  * y            (y) float64 808kB 1.8e+06 1.8e+06 ... 7.905e+05 7.905e+05
  * x            (x) float64 616kB 1.38e+05 1.38e+05 ... 9.083e+05 9.083e+05
  * time         (time) datetime64[us] 8B 2020-06-01
    spatial_ref  int32 4B 32651
Data variables:
    data         (time, y, x) float32 31GB dask.array<chunksize=(1, 1024, 1024), meta=np.ndarray>